In [ ]:
"""
add_more_plates_to_gel.py

Adds rigid graphene-like plates to ALL SIX faces of an isolated gel data file:
  - Top plate    (+z face) : xy-plane lattice, spanning full box in xy
  - Bottom plate (-z face) : xy-plane lattice, spanning full box in xy
  - Front plate  (+y face) : xz-plane lattice, spanning full box in x, gel z-extent
  - Back plate   (-y face) : xz-plane lattice, spanning full box in x, gel z-extent
  - Right plate  (+x face) : yz-plane lattice, spanning full box in y, gel z-extent
  - Left plate   (-x face) : yz-plane lattice, spanning full box in y, gel z-extent

Design
------
- Plate atoms are atom type 4 (mass 1.0), arranged on a square lattice.
- NO bonds are created between plate atoms and the polymer network.
  Contact is purely through WCA (LJ) interactions. This avoids lateral
  forces on surface chains when the plates compress the gel — bonded
  plate atoms would stretch sideways as the plate moves in z, transmitting
  unphysical shear to the network edges.
- Output data file has 4 atom types and 1 bond type (FENE only, unchanged).

Plate-Plate Interactions
------------------------
Plate-plate interactions MUST be disabled in the LAMMPS input script.
Recommended approach (add to LAMMPS input):
    neigh_modify exclude type 4 4
Alternative (zero out epsilon):
    pair_coeff 4 4 0.0 1.0 0.0

Bond style needed in LAMMPS input (unchanged from gel):
    bond_style fene
    bond_coeff 1 30.0 1.5 1.0 1.0   # polymer-polymer (FENE)

Pair coeffs for plate (type 4) — add to LAMMPS input:
    pair_coeff 4 4 0.0 1.0 0.0      # plate-plate  DISABLED
    pair_coeff 1 4 1.0 1.0 1.122    # polymer-plate (WCA)
    pair_coeff 2 4 1.0 1.0 1.122    # crosslinker-plate (WCA)
    pair_coeff 3 4 1.0 1.0 1.122    # solvent-plate (WCA)
"""
#Test 

import numpy as np

# ── Defaults ─────────────────────────────────────────────────────────────────
PLATE_TYPE      = 4      # new atom type for plate beads
PLATE_SPACING   = 0.2    # square lattice constant (σ)
PLATE_OFFSET    = 0.5    # distance from polymer surface (σ)
SURFACE_PCT     = 1.0    # percentile to robustly define surface (avoids outliers)

POLYMER_TYPES   = {1, 2} # atom types used for surface detection only


# ── I/O helpers ──────────────────────────────────────────────────────────────

def parse_lammps_data(filename):
    """Return atoms, bonds, box_bounds, masses from a LAMMPS molecular data file."""
    atoms, bonds, masses = [], [], {}
    box = {}

    with open(filename) as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if 'xlo xhi' in line:
            p = line.split(); box['xlo'], box['xhi'] = float(p[0]), float(p[1])
        elif 'ylo yhi' in line:
            p = line.split(); box['ylo'], box['yhi'] = float(p[0]), float(p[1])
        elif 'zlo zhi' in line:
            p = line.split(); box['zlo'], box['zhi'] = float(p[0]), float(p[1])

        elif line == 'Masses':
            i += 2
            while i < len(lines) and lines[i].strip() and \
                    not lines[i].strip()[0].isalpha():
                p = lines[i].split()
                if len(p) >= 2:
                    masses[int(p[0])] = float(p[1])
                i += 1
            continue

        elif line.startswith('Atoms'):
            i += 2
            while i < len(lines) and lines[i].strip() and \
                    not lines[i].strip()[0].isalpha():
                p = lines[i].split()
                if len(p) >= 6:
                    atoms.append({
                        'id':   int(p[0]),
                        'mol':  int(p[1]),
                        'type': int(p[2]),
                        'x':    float(p[3]),
                        'y':    float(p[4]),
                        'z':    float(p[5]),
                    })
                i += 1
            continue

        elif line.startswith('Bonds'):
            i += 2
            while i < len(lines) and lines[i].strip() and \
                    not lines[i].strip()[0].isalpha():
                p = lines[i].split()
                if len(p) >= 4:
                    bonds.append({
                        'id':    int(p[0]),
                        'type':  int(p[1]),
                        'atom1': int(p[2]),
                        'atom2': int(p[3]),
                    })
                i += 1
            continue

        i += 1

    return atoms, bonds, box, masses


def write_lammps_data(filename, atoms, bonds, box, masses):
    """Write LAMMPS molecular data file. Atom IDs are renumbered 1..N."""
    old2new = {a['id']: i + 1 for i, a in enumerate(atoms)}

    valid_bonds = []
    for i, b in enumerate(bonds):
        a1 = old2new.get(b['atom1'])
        a2 = old2new.get(b['atom2'])
        if a1 and a2:
            valid_bonds.append({'id': i + 1, 'type': b['type'],
                                 'atom1': a1, 'atom2': a2})

    n_types  = max(a['type'] for a in atoms)
    n_btypes = max((b['type'] for b in valid_bonds), default=1)

    with open(filename, 'w') as f:
        f.write("LAMMPS data file - isolated gel with six shear plates\n\n")
        f.write(f"{len(atoms)} atoms\n")
        f.write(f"{len(valid_bonds)} bonds\n\n")
        f.write(f"{n_types} atom types\n")
        f.write(f"{n_btypes} bond types\n\n")
        f.write(f"{box['xlo']:.6f} {box['xhi']:.6f} xlo xhi\n")
        f.write(f"{box['ylo']:.6f} {box['yhi']:.6f} ylo yhi\n")
        f.write(f"{box['zlo']:.6f} {box['zhi']:.6f} zlo zhi\n\n")

        f.write("Masses\n\n")
        for t in range(1, n_types + 1):
            f.write(f"{t} {masses.get(t, 1.0):.4f}\n")

        f.write("\nAtoms\n\n")
        for i, a in enumerate(atoms, 1):
            f.write(f"{i} {a['mol']} {a['type']} "
                    f"{a['x']:.6f} {a['y']:.6f} {a['z']:.6f}\n")

        if valid_bonds:
            f.write("\nBonds\n\n")
            for b in valid_bonds:
                f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")


# ── Surface detection ─────────────────────────────────────────────────────────

def surface_coord(coords: np.ndarray, top: bool, pct: float) -> float:
    """Robust surface coordinate using percentile (avoids outlier atoms)."""
    return float(np.percentile(coords, 100 - pct if top else pct))


# ── Plate generation ──────────────────────────────────────────────────────────

def make_plate_xy(box, z_plane, spacing, mol_id):
    """
    Top or bottom plate: square lattice in the xy-plane at z = z_plane.
    Spans [xlo, xhi) x [ylo, yhi) — tiles perfectly under PBC.
    """
    lx = box['xhi'] - box['xlo']
    ly = box['yhi'] - box['ylo']

    nx = int(np.floor(lx / spacing))
    ny = int(np.floor(ly / spacing))

    dx = lx / nx
    dy = ly / ny
    x0 = box['xlo'] + dx / 2.0
    y0 = box['ylo'] + dy / 2.0

    plate = []
    for ix in range(nx):
        for iy in range(ny):
            plate.append({
                'id':   None,
                'mol':  mol_id,
                'type': PLATE_TYPE,
                'x':    x0 + ix * dx,
                'y':    y0 + iy * dy,
                'z':    z_plane,
            })
    return plate


def make_plate_xz(box, y_plane, z_lo, z_hi, spacing, mol_id):
    """
    Front (+y) or back (-y) plate: square lattice in the xz-plane at y = y_plane.
    Spans [xlo, xhi) in x (tiles under PBC) and [z_lo, z_hi] in z (gel z-extent).
    """
    lx = box['xhi'] - box['xlo']
    lz = z_hi - z_lo

    nx = int(np.floor(lx / spacing))
    nz = max(1, int(np.round(lz / spacing)))

    dx = lx / nx
    dz = lz / nz if nz > 1 else spacing
    x0 = box['xlo'] + dx / 2.0
    z0 = z_lo + dz / 2.0

    plate = []
    for ix in range(nx):
        for iz in range(nz):
            plate.append({
                'id':   None,
                'mol':  mol_id,
                'type': PLATE_TYPE,
                'x':    x0 + ix * dx,
                'y':    y_plane,
                'z':    z0 + iz * dz,
            })
    return plate


def make_plate_yz(box, x_plane, z_lo, z_hi, spacing, mol_id):
    """
    Right (+x) or left (-x) plate: square lattice in the yz-plane at x = x_plane.
    Spans [ylo, yhi) in y (tiles under PBC) and [z_lo, z_hi] in z (gel z-extent).
    """
    ly = box['yhi'] - box['ylo']
    lz = z_hi - z_lo

    ny = int(np.floor(ly / spacing))
    nz = max(1, int(np.round(lz / spacing)))

    dy = ly / ny
    dz = lz / nz if nz > 1 else spacing
    y0 = box['ylo'] + dy / 2.0
    z0 = z_lo + dz / 2.0

    plate = []
    for iy in range(ny):
        for iz in range(nz):
            plate.append({
                'id':   None,
                'mol':  mol_id,
                'type': PLATE_TYPE,
                'x':    x_plane,
                'y':    y0 + iy * dy,
                'z':    z0 + iz * dz,
            })
    return plate


# ── Main ─────────────────────────────────────────────────────────────────────

def add_six_plates(input_file, output_file,
                   spacing=PLATE_SPACING, offset=PLATE_OFFSET,
                   surface_pct=SURFACE_PCT):

    print("=" * 60)
    print(f"add_more_plates_to_gel.py")
    print(f"  Input  : {input_file}")
    print(f"  Output : {output_file}")
    print(f"  Lattice spacing : {spacing} σ")
    print(f"  Plate offset    : {offset} σ from gel surface")
    print(f"  Bond mode       : NONE (pure WCA contact only)")
    print(f"  Plates: top (+z), bottom (-z), front (+y), back (-y), right (+x), left (-x)")
    print("=" * 60)

    atoms, bonds, box, masses = parse_lammps_data(input_file)
    print(f"Read {len(atoms)} atoms, {len(bonds)} bonds")

    # ── Identify polymer atoms for surface detection ──────────────────────────
    poly_all = [a for a in atoms if a['type'] in POLYMER_TYPES]
    poly_x   = np.array([a['x'] for a in poly_all])
    poly_y   = np.array([a['y'] for a in poly_all])
    poly_z   = np.array([a['z'] for a in poly_all])

    # Z surfaces (top/bottom plates)
    z_top_surf  = surface_coord(poly_z, top=True,  pct=surface_pct)
    z_bot_surf  = surface_coord(poly_z, top=False, pct=surface_pct)
    gel_thick_z = z_top_surf - z_bot_surf

    # Y surfaces (front/back plates)
    y_front_surf = surface_coord(poly_y, top=True,  pct=surface_pct)
    y_back_surf  = surface_coord(poly_y, top=False, pct=surface_pct)
    gel_thick_y  = y_front_surf - y_back_surf

    # X surfaces (right/left plates)
    x_right_surf = surface_coord(poly_x, top=True,  pct=surface_pct)
    x_left_surf  = surface_coord(poly_x, top=False, pct=surface_pct)
    gel_thick_x  = x_right_surf - x_left_surf

    print(f"\nGel z-extent : {z_bot_surf:.3f} → {z_top_surf:.3f}  "
          f"(thickness {gel_thick_z:.2f} σ)")
    print(f"Gel y-extent : {y_back_surf:.3f} → {y_front_surf:.3f}  "
          f"(thickness {gel_thick_y:.2f} σ)")
    print(f"Gel x-extent : {x_left_surf:.3f} → {x_right_surf:.3f}  "
          f"(thickness {gel_thick_x:.2f} σ)")

    # ── Generate plate atoms ──────────────────────────────────────────────────
    z_top_plate   = z_top_surf   + offset
    z_bot_plate   = z_bot_surf   - offset
    y_front_plate = y_front_surf + offset
    y_back_plate  = y_back_surf  - offset
    x_right_plate = x_right_surf + offset
    x_left_plate  = x_left_surf  - offset

    max_mol = max(a['mol'] for a in atoms)
    plate_top   = make_plate_xy(box, z_top_plate,   spacing, mol_id=max_mol + 1)
    plate_bot   = make_plate_xy(box, z_bot_plate,   spacing, mol_id=max_mol + 2)
    plate_front = make_plate_xz(box, y_front_plate,
                                 z_bot_surf, z_top_surf, spacing, mol_id=max_mol + 3)
    plate_back  = make_plate_xz(box, y_back_plate,
                                 z_bot_surf, z_top_surf, spacing, mol_id=max_mol + 4)
    plate_right = make_plate_yz(box, x_right_plate,
                                 z_bot_surf, z_top_surf, spacing, mol_id=max_mol + 5)
    plate_left  = make_plate_yz(box, x_left_plate,
                                 z_bot_surf, z_top_surf, spacing, mol_id=max_mol + 6)

    print(f"\nPlate positions and sizes:")
    print(f"  Top plate   (+z): z = {z_top_plate:.3f}  |  {len(plate_top)} atoms")
    print(f"  Bottom plate(-z): z = {z_bot_plate:.3f}  |  {len(plate_bot)} atoms")
    print(f"  Front plate (+y): y = {y_front_plate:.3f}  |  {len(plate_front)} atoms")
    print(f"  Back plate  (-y): y = {y_back_plate:.3f}  |  {len(plate_back)} atoms")
    print(f"  Right plate (+x): x = {x_right_plate:.3f}  |  {len(plate_right)} atoms")
    print(f"  Left plate  (-x): x = {x_left_plate:.3f}  |  {len(plate_left)} atoms")

    # ── Assign IDs ────────────────────────────────────────────────────────────
    next_id = max(a['id'] for a in atoms) + 1
    for a in (plate_top + plate_bot + plate_front + plate_back
              + plate_right + plate_left):
        a['id'] = next_id
        next_id += 1

    masses[PLATE_TYPE] = 1.0

    # ── Extend box to contain all plates (0.5σ buffer) ────────────────────────
    new_box = dict(box)
    new_box['zlo'] = min(box['zlo'], z_bot_plate   - 0.5)
    new_box['zhi'] = max(box['zhi'], z_top_plate   + 0.5)
    new_box['ylo'] = min(box['ylo'], y_back_plate  - 0.5)
    new_box['yhi'] = max(box['yhi'], y_front_plate + 0.5)
    new_box['xlo'] = min(box['xlo'], x_left_plate  - 0.5)
    new_box['xhi'] = max(box['xhi'], x_right_plate + 0.5)

    # ── Assemble and write ────────────────────────────────────────────────────
    all_atoms = (atoms + plate_top + plate_bot + plate_front
                 + plate_back + plate_right + plate_left)

    write_lammps_data(output_file, all_atoms, bonds, new_box, masses)

    n_plate = (len(plate_top) + len(plate_bot) + len(plate_front)
               + len(plate_back) + len(plate_right) + len(plate_left))
    print(f"\nWrote {len(all_atoms)} atoms ({n_plate} plate atoms), "
          f"{len(bonds)} bonds (unchanged) to:\n  {output_file}")
    print("=" * 60)

    # ── Summary for LAMMPS input script ──────────────────────────────────────
    print("\n── LAMMPS input script hints ────────────────────────────────")
    print(f"  gel z-thickness (surface-to-surface) ≈ {gel_thick_z:.2f} σ")
    print(f"  gel y-thickness (surface-to-surface) ≈ {gel_thick_y:.2f} σ")
    print(f"  gel x-thickness (surface-to-surface) ≈ {gel_thick_x:.2f} σ")
    print(f"  top/bot plate separation             ≈ {z_top_plate - z_bot_plate:.2f} σ")
    print(f"  front/back plate separation          ≈ {y_front_plate - y_back_plate:.2f} σ")
    print(f"  right/left plate separation          ≈ {x_right_plate - x_left_plate:.2f} σ")
    print()
    print("  # ── Plate-plate interactions MUST be disabled ──")
    print("  neigh_modify exclude type 4 4        # recommended")
    print("  # OR: pair_coeff 4 4 0.0 1.0 0.0    # zero epsilon")
    print()
    print("  bond_style fene")
    print("  bond_coeff 1 30.0 1.5 1.0 1.0        # polymer-polymer (unchanged)")
    print()
    print("  pair_coeff 4 4 0.0 1.0 0.0            # plate-plate  DISABLED")
    print("  pair_coeff 1 4 1.0 1.0 1.122          # polymer-plate (WCA)")
    print("  pair_coeff 2 4 1.0 1.0 1.122          # crosslinker-plate (WCA)")
    print("  pair_coeff 3 4 1.0 1.0 1.122          # solvent-plate (WCA)")
    print("─" * 60)


In [ ]:
# Inputs

input_file  = "../../lammps_data_files_local/isolated_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000.data"
output_file = "../../lammps_data_files_local/isolated_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600001_with_six_plates.data"

add_six_plates(input_file, output_file)